# Gas-turbine combustor (reacting mean flow)

A complete combustor as an Nefes network: **compressor-discharge air** -> a
**fuel mass source** (the injector) -> an **equilibrium flame** (ignition) ->
a **dilution-air mass source** (cooling to the turbine-inlet temperature) ->
the **turbine-inlet** outlet.

Key ideas exercised here:

* the thermo data is read from the NASA Glenn / CEA `thermo.inp` database via
  the standalone `thermolib`; you pick the species you need;
* streams are specified by **species** name (`{'O2': 0.21, 'N2': 0.79}`); the
  network transports one conserved **mixture fraction** per distinct injected
  composition (here: air and fuel), discovered automatically at build time;
* the **mass source** adds mass / momentum / energy with the proper source
  terms -- a fuel injector is just a mass source that injects fuel;
* **ignition** is done only by the flame element (frozen unburnt -> equilibrium
  burnt); the injector never reacts.

Tweak `mdot_fuel`, `mdot_dilution`, `Tair`, `p` in the builder and re-run.

In [ ]:
import os, sys
# add the repo root (the dir containing the `nefes` package) to sys.path
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import matplotlib.pyplot as plt

from thermolib import ThermoInp, Thermo
from nefes.chem.composition import resolve_composition, enthalpy_mass
from nefes.elements import catalog as cat
from nefes.solver import solve
from nefes.solver.control import states_table
from nefes.thermo.api import EQ_FROZEN, EQ_KERNEL
from nefes.thermo.configure import equilibrium
from nefes.assembly.derive import ES_MDOT, ES_P, ES_HT, ES_RHO, ES_U, ES_T, ES_M, ES_PT
from nefes.shell import Network

THERMO_INP = os.path.join(_root, "thermolib", "data", "thermo.inp")
RU = 8.31446261815324

## Species library
A small CH4/air library selected from `thermo.inp`: the air species and the
fuel, plus the major combustion products the equilibrium solver fills in.

In [ ]:
lib = ThermoInp(THERMO_INP).library(
    ["O2", "N2", "Ar", "CO2", "CH4", "H2O", "CO", "OH", "H", "O", "NO", "H2"]
)
gas = Thermo(lib)
print("elements:", lib.elements)
print("species :", [s.name for s in lib.species])

## Build the combustor network
The helper assembles the five elements and the four edges.  `solve(prob)` seeds
itself by **propagating the feeds through the network** (mass-weighted mixing of
each stream's `(mdot, h_t, xi)`), so no hand-built per-edge guess is needed --
even though the network spans a wide enthalpy range (cold air vs hot products vs
CH4's strongly negative formation enthalpy).

In [ ]:
AIR = {"O2": 0.2095, "N2": 0.7808, "Ar": 0.0093, "CO2": 0.0004}
FUEL = {"CH4": 1.0}

def build_combustor(mdot_air=1.0, mdot_fuel=0.045, mdot_dil=1.0, Tair=700.0, p=10.0e5, A=0.05):
    """air-in -> fuel injector -> flame -> dilution -> turbine-in.

    Air and fuel are the two feed streams (the dilution injects the same air, so it
    shares the air mixture fraction).  They are discovered from the element
    compositions at build time; `solve(prob)` then seeds every edge by propagating
    the feeds through the network -- no hand-built guess.
    """
    Yair, _ = resolve_composition(lib, AIR)
    Yfuel, _ = resolve_composition(lib, FUEL)
    h_air = enthalpy_mass(lib, Yair, Tair)
    h_fuel = enthalpy_mass(lib, Yfuel, Tair)
    m1 = mdot_air + mdot_fuel                       # after the injector
    m2 = m1 + mdot_dil                              # after dilution
    h_mix = (mdot_air * h_air + mdot_fuel * h_fuel) / m1  # enthalpy scale for h_ref

    cfg = equilibrium(lib)
    els = [
        cat.mass_flow_inlet(mdot_air, Tair, composition=AIR, name="air-in"),
        cat.mass_source(mdot_fuel, Tair, composition=FUEL, name="fuel"),
        cat.equilibrium_flame(name="flame"),
        cat.mass_source(mdot_dil, Tair, composition=AIR, name="dilution"),
        cat.pressure_outlet(p, Tt_backflow=Tair, composition=AIR, name="turbine-in"),
    ]
    edges = [(0, 1, A), (1, 2, A), (2, 3, A), (3, 4, A)]
    edge_models = [EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL]
    network = Network(cfg, p_ref=p, mdot_ref=m2, h_ref=abs(h_mix), nodes=els, edges=edges,
                      edge_models=edge_models)
    prob = cat.build_problem(cfg, els, edges, mdot_ref=m2, p_ref=p, h_ref=abs(h_mix),
                             edge_models=edge_models)
    return cfg, prob, network

cfg, prob, network = build_combustor()
res = solve(prob)
print("converged:", res.converged, " Newton iterations:", res.iterations)

## Converged edge states

In [ ]:
est = states_table(prob, res.x)
names = ["air", "air+fuel", "burnt", "diluted"]
print(f"{'edge':<10}{'mdot':>9}{'T [K]':>10}{'p [bar]':>10}{'M':>8}{'h_t [J/kg]':>14}")
for e in range(prob.n_edges):
    print(f"{names[e]:<10}{est[ES_MDOT,e]:9.4f}{est[ES_T,e]:10.1f}"
          f"{est[ES_P,e]/1e5:10.3f}{est[ES_M,e]:8.3f}{est[ES_HT,e]:14.3e}")

print()
print(f"global mass in  : {est[ES_MDOT,0]:.4f} kg/s (air) + injected fuel + dilution")
print(f"global mass out : {est[ES_MDOT,3]:.4f} kg/s")
print(f"flame conserves h_t: {est[ES_HT,1]:.4e} -> {est[ES_HT,2]:.4e} J/kg")

## Axial temperature profile
The flame jump, then dilution cooling to the turbine-inlet temperature.

In [ ]:
x_ax = [0, 1, 2, 3]
T = [est[ES_T, e] for e in range(4)]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_ax, T, "o-", lw=2, ms=8, color="#c0392b")
for xi, ti, nm in zip(x_ax, T, names):
    ax.annotate(f"{nm}\n{ti:.0f} K", (xi, ti), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9)
ax.set_xticks(x_ax); ax.set_xticklabels(["air-in", "injector", "flame", "turbine-in"])
ax.set_ylabel("static temperature [K]"); ax.set_title("Combustor axial temperature")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Fuel-flow sweep: equivalence ratio -> flame & turbine-inlet temperature
With the dilution flow fixed, raising the fuel flow raises the primary flame
temperature and the diluted turbine-inlet temperature.

In [ ]:
# stoichiometric CH4/air mass ratio (for the equivalence ratio of the primary zone)
f_stoich = 0.05518
fuels = np.linspace(0.02, 0.07, 11)
Tflame, Tturb, phi = [], [], []
for mf in fuels:
    cfg_i, prob_i, _ = build_combustor(mdot_fuel=mf)
    r = solve(prob_i)
    e = states_table(prob_i, r.x)
    Tflame.append(e[ES_T, 2]); Tturb.append(e[ES_T, 3])
    phi.append((mf / 1.0) / f_stoich)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(phi, Tflame, "o-", label="flame (primary)", color="#c0392b")
ax.plot(phi, Tturb, "s-", label="turbine-inlet (diluted)", color="#2980b9")
ax.set_xlabel("primary equivalence ratio  phi"); ax.set_ylabel("temperature [K]")
ax.set_title("Fuel-flow sweep"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Dilution sweep: trading dilution air for turbine-inlet temperature
More dilution air cools the products toward the metallurgical turbine-inlet limit.

In [ ]:
dils = np.linspace(0.3, 2.0, 11)
Tturb_d = []
for md in dils:
    cfg_i, prob_i, _ = build_combustor(mdot_dil=md)
    r = solve(prob_i)
    Tturb_d.append(states_table(prob_i, r.x)[ES_T, 3])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dils, Tturb_d, "o-", color="#16a085")
ax.set_xlabel("dilution air mdot [kg/s]"); ax.set_ylabel("turbine-inlet T [K]")
ax.set_title("Dilution sweep"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Export for FNetLibUI

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.save(path)` embeds the mean-flow result fields, `network.save(path)` writes the topology only — then open the file in FNetLibUI.

In [ ]:
import os, tempfile

sol = network.solve()  # the primary combustor network, re-solved via the shell
_out = os.path.join(tempfile.mkdtemp(), "gas_turbine_combustor.yaml")
sol.save(_out)  # embeds the mean-flow results; use network.save(_out) for topology only
print("saved case:", _out)